<a href="https://colab.research.google.com/github/abdulhadi2005ag-cmd/flyrank-ml-internship-hadii/blob/main/work/notebooks/Copy_of_w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulhadi2005ag-cmd/flyrank-ml-internship-hadii/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/abdulhadi2005ag-cmd/flyrank-ml-internship-hadii"
REPO_DIR = "flyrank-ml-internship-hadii"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Loaded:", df.shape)

Loaded: (30000, 44)


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My rule idea for Lane 2: a page is worth an editor's time if it's **stale**, still gets **real traffic**,
and its **click-through is weak for its rank** (a title/meta fix would likely help). Before coding that,
I check the two signals it leans on.

**Signal 1 — staleness (`days_since_last_update`), the signal behind the refresh flags.**
Claim: pages that haven't been touched in a while should show up more often in the "declining" bucket
(`trend_direction == "down"`, used here only as a check target, never as a rule input — it's the kind of
label-derived column the data skill flags as never-a-feature).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
bins = [0, 30, 90, 180, 10_000]
labels = ["0-30", "31-90", "91-180", "181+"]
df["staleness_bucket"] = pd.cut(df["days_since_last_update"], bins=bins, labels=labels)

staleness_table = (
    df.groupby("staleness_bucket", observed=True)
      .agg(n=("content_id", "size"),
           decline_rate=("trend_direction", lambda s: (s == "down").mean()))
      .reset_index()
)
print(staleness_table.to_string(index=False))
print()
print("overall decline rate (base rate):", round((df["trend_direction"] == "down").mean(), 3))


staleness_bucket     n  decline_rate
            0-30 20480      0.511377
           31-90   175      0.588571
          91-180  9171      0.611057
            181+   174      0.471264

overall decline rate (base rate): 0.542


**Verdict: MIXED.** Decline rate climbs from 51% (0-30 days) to 61% (91-180 days) — staleness does
track with decline, as the refresh flags assume. But the 181+ bucket drops back to 47%, *below* the
overall base rate (54%) — and that bucket only has n=174, a tenth the size of the others. I read this as:
staleness is a real signal up to ~180 days, but I don't trust the reversal at 181+ enough to act on it —
it's more likely a small, noisy bucket than proof that very old pages are fine. My rule will use staleness
as a simple yes/no flag (≥90 days), not reward "older is always better."

**Signal 2 — CTR vs. position (`position_tier`), the signal behind the CTR-fix logic.**
Claim: better search position should mean higher click-through — if it doesn't, that page is a CTR-fix
candidate (title/meta), not a content problem.

In [ ]:
order = ["top_3", "page_1", "page_3_5", "striking", "deep"]

# First pass — everyone, including avg_position == 0
raw_table = (
    df.groupby("position_tier", observed=True)
      .agg(n=("content_id", "size"), mean_ctr=("ctr", "mean"), median_ctr=("ctr", "median"))
      .reindex(order)
)
print("Raw (includes avg_position == 0, which the data dictionary says means NO DATA, not rank zero):")
print(raw_table.to_string())
print()
print("rows with avg_position == 0:", (df["avg_position"] == 0).sum(),
      "— all land in position_tier:", df.loc[df["avg_position"] == 0, "position_tier"].unique())

Raw (includes avg_position == 0, which the data dictionary says means NO DATA, not rank zero):
                   n  mean_ctr  median_ctr
position_tier                             
top_3           2321  1.483611        0.00
page_1         11814  0.652467        0.16
page_3_5        7242  0.222484        0.03
striking        7304  0.323239        0.11
deep            1319  0.150212        0.00

rows with avg_position == 0: 1205 — all land in position_tier: ['top_3']


In [ ]:
# Second pass — drop the no-data rows before judging the signal
clean = df[df["avg_position"] > 0]
ctr_table = (
    clean.groupby("position_tier", observed=True)
         .agg(n=("content_id", "size"), mean_ctr=("ctr", "mean"), median_ctr=("ctr", "median"))
         .reindex(order)
)
print("Cleaned (avg_position > 0 only):")
print(ctr_table.to_string())

Cleaned (avg_position > 0 only):
                   n  mean_ctr  median_ctr
position_tier                             
top_3           1116  2.764453        0.00
page_1         11814  0.652467        0.16
page_3_5        7242  0.222484        0.03
striking        7304  0.323239        0.11
deep            1319  0.150212        0.00


**Verdict: CONFIRMED.** The raw table looked broken — `top_3` had the *worst*-looking CTR of any tier.
The reason: all 1,205 rows with `avg_position == 0` (the dataset's "no data" marker, per the data
dictionary) get bucketed into `top_3` by construction, dragging its average down with untracked pages.
Once I drop those, `top_3` has the highest mean CTR (2.76) and the ranking runs mostly top_3 > page_1 >
striking > page_3_5 > deep, matching the CTR-fix assumption. Median CTR is still 0 for `top_3` and `deep`
(the click distribution is heavily skewed/zero-inflated), so I'll compare each page's CTR against its own
tier's **median**, not the mean, when flagging weak CTR — the median is what "typical" looks like once the
skew is accounted for.

**My rule, in plain words:** flag a page for refresh review if (1) it hasn't been updated in 90+ days,
(2) it still gets meaningful real traffic (≥500 impressions in 90 days, so I'm not spending review time on
dead pages), and (3) its CTR is below the median CTR for pages at its own position tier (a real,
tier-adjusted underperformance, not just "low position = low CTR"). Score = impressions_90d for anything
that clears all three gates, 0 otherwise — bigger audience behind the flag means it's ranked higher.

**Reason code (one, constant for every flagged row):** `stale_weak_ctr_with_demand`
**Action label:** `refresh_priority` if flagged, else `no_action`

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Tier-median CTR benchmark, built only from rows with real position data
tier_median_ctr = clean.groupby("position_tier", observed=True)["ctr"].median()
print("Tier median CTR benchmark:")
print(tier_median_ctr.to_string())

df["has_position_data"] = df["avg_position"] > 0
df["tier_median_ctr"] = df["position_tier"].map(tier_median_ctr)

# 1) stale — hasn't been touched in 90+ days
df["stale"] = df["days_since_last_update"] >= 90

# 2) has_demand — still gets real traffic, so review effort isn't wasted on a dead page
df["has_demand"] = df["impressions_90d"] >= 500

# 3) weak_ctr — CTR below the median for its OWN position tier, and we trust the tier (real position data)
df["weak_ctr"] = df["has_position_data"] & (df["ctr"] < df["tier_median_ctr"])

flagged = df["stale"] & df["has_demand"] & df["weak_ctr"]

df["reason_code"] = np.where(flagged, "stale_weak_ctr_with_demand", "not_flagged")
df["action"] = np.where(flagged, "refresh_priority", "no_action")
df["score"] = np.where(flagged, df["impressions_90d"], 0)

print()
print("flagged rows:", int(flagged.sum()), "of", len(df), f"({flagged.mean():.1%})")


Tier median CTR benchmark:
position_tier
deep        0.00
page_1      0.16
page_3_5    0.03
striking    0.11
top_3       0.00

flagged rows: 2019 of 30000 (6.7%)


In [ ]:
queue = (
    df[["content_id", "client_id", "score", "reason_code", "action",
        "days_since_last_update", "impressions_90d", "ctr", "position_tier", "avg_position"]]
    .sort_values("score", ascending=False)
    .reset_index(drop=True)
)

os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Wrote", len(queue), "rows to work/outputs/baseline_action_score.csv")
queue.head(10)

Wrote 30000 rows to work/outputs/baseline_action_score.csv


,content_id,client_id,score,reason_code,action,days_since_last_update,impressions_90d,ctr,position_tier,avg_position
0,content_5fe46e04994d,client_4e07408562,517715,stale_weak_ctr_with_demand,refresh_priority,104,517715,0.14,page_1,4.2
1,content_36ff89c8214e,client_19581e27de,295097,stale_weak_ctr_with_demand,refresh_priority,104,295097,0.05,page_1,7.3
2,content_c8e9d6ab9013,client_19581e27de,208678,stale_weak_ctr_with_demand,refresh_priority,104,208678,0.00,page_1,9.7
3,content_a7427266c305,client_19581e27de,201111,stale_weak_ctr_with_demand,refresh_priority,104,201111,0.11,page_1,5.7
4,content_91652435f57a,client_19581e27de,159590,stale_weak_ctr_with_demand,refresh_priority,104,159590,0.06,page_1,7.8
5,content_f42eb861c6dd,client_19581e27de,152467,stale_weak_ctr_with_demand,refresh_priority,104,152467,0.13,page_1,6.5
6,content_11fcfd65d94c,client_19581e27de,149083,stale_weak_ctr_with_demand,refresh_priority,104,149083,0.15,page_1,6.2
7,content_97a86caf3a3d,client_19581e27de,147670,stale_weak_ctr_with_demand,refresh_priority,104,147670,0.07,page_1,6.4
8,content_8b36799b7e44,client_6208ef0f77,141400,stale_weak_ctr_with_demand,refresh_priority,104,141400,0.02,page_3_5,32.0
9,content_c1fe78bc4e37,client_19581e27de,134055,stale_weak_ctr_with_demand,refresh_priority,104,134055,0.03,page_1,7.5


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top10 = queue.head(10).reset_index(drop=True)
for i, row in top10.iterrows():
    print(f"{i+1}. {row['content_id']} — action: {row['action']} (score={int(row['score']):,})")
    print(f"   Why: {row['days_since_last_update']} days since last update (stale), "
          f"{int(row['impressions_90d']):,} impressions/90d (real demand), "
          f"CTR {row['ctr']} vs its {row['position_tier']} tier — flagged as an underperforming click-through.")
    print(f"   Would be wrong if: the page was refreshed or its title/meta was already changed very "
          f"recently and this snapshot hasn't caught up, or if {row['position_tier']} is itself a "
          f"low-CTR-by-nature tier (e.g. a featured snippet position) where low CTR isn't fixable by a rewrite.")
    print()

1. content_5fe46e04994d — action: refresh_priority (score=517,715)
   Why: 104 days since last update (stale), 517,715 impressions/90d (real demand), CTR 0.14 vs its page_1 tier — flagged as an underperforming click-through.
   Would be wrong if: the page was refreshed or its title/meta was already changed very recently and this snapshot hasn't caught up, or if page_1 is itself a low-CTR-by-nature tier (e.g. a featured snippet position) where low CTR isn't fixable by a rewrite.

2. content_36ff89c8214e — action: refresh_priority (score=295,097)
   Why: 104 days since last update (stale), 295,097 impressions/90d (real demand), CTR 0.05 vs its page_1 tier — flagged as an underperforming click-through.
   Would be wrong if: the page was refreshed or its title/meta was already changed very recently and this snapshot hasn't caught up, or if page_1 is itself a low-CTR-by-nature tier (e.g. a featured snippet position) where low CTR isn't fixable by a rewrite.

3. content_c8e9d6ab9013 — action

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
weak = queue.head(10).merge(
    tier_median_ctr.rename("tier_median_ctr"), left_on="position_tier", right_index=True
)
weak["ctr_gap"] = weak["tier_median_ctr"] - weak["ctr"]
print(weak[["content_id", "position_tier", "ctr", "tier_median_ctr", "ctr_gap"]].to_string(index=False))
print()
print("Weakest pick: the row with the smallest ctr_gap — its CTR is barely below its tier's median,")
print("so it's a marginal call, not a clear underperformer. Worth a lower-confidence flag in practice.")

# Leakage check: every input to the score is knowable before any decision point —
# no trend_direction/trend_pct (label-derived, per the data skill), no future 30-day window
score_inputs = ["days_since_last_update", "impressions_90d", "ctr", "avg_position", "position_tier"]
leak_cols = {"trend_direction", "trend_pct", "is_declining_label"}
print()
print("Score inputs:", score_inputs)
print("Any leak-listed column used in the score?", any(c in leak_cols for c in score_inputs))


          content_id position_tier  ctr  tier_median_ctr  ctr_gap
content_5fe46e04994d        page_1 0.14             0.16     0.02
content_36ff89c8214e        page_1 0.05             0.16     0.11
content_c8e9d6ab9013        page_1 0.00             0.16     0.16
content_a7427266c305        page_1 0.11             0.16     0.05
content_91652435f57a        page_1 0.06             0.16     0.10
content_f42eb861c6dd        page_1 0.13             0.16     0.03
content_11fcfd65d94c        page_1 0.15             0.16     0.01
content_97a86caf3a3d        page_1 0.07             0.16     0.09
content_8b36799b7e44      page_3_5 0.02             0.03     0.01
content_c1fe78bc4e37        page_1 0.03             0.16     0.13

Weakest pick: the row with the smallest ctr_gap — its CTR is barely below its tier's median,
so it's a marginal call, not a clear underperformer. Worth a lower-confidence flag in practice.

Score inputs: ['days_since_last_update', 'impressions_90d', 'ctr', 'avg_position', 

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.